In [ ]:
!pip install qdrant-client sentence-transformers transformers torch

  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cusparse_cu12-12.1.0.106-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_nccl_cu12-2.20.5-py3-none-manylinux2014_x86_64.whl.metadata (1.8 kB)
  Using cached nvidia_nvtx_cu12-12.1.105-py3-none-manylinu

In [ ]:
from qdrant_client import QdrantClient, models
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

/usr/local/lib/python3.10/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [ ]:
# Qdrant 클라이언트 초기화
QDRANT_URL = "https://6e46b2c2-f28a-4f28-854d-432ab699fdfd.europe-west3-0.gcp.cloud.qdrant.io"
QDRANT_API_KEY = "u2eejPgTwIyhr7BVjFBtkjGdGYPWvzQTBkoYycErtm5cyrFjwEEH9w"
client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

# 임베딩 모델 초기화
encoder = SentenceTransformer("jhgan/ko-sroberta-multitask")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/4.86k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/585 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/495k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# ko-gpt-trinity-cw 모델 초기화
model_name = "centwon/ko-gpt-trinity-cw"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

def search_disease_info(query_text, limit=3):
    query_vector = encoder.encode(query_text).tolist()
    search_results = client.search(
        collection_name='son5',
        query_vector=query_vector,
        limit=limit
    )
    results = []
    for hit in search_results:
        result = {
            "질병": hit.payload.get("질병", "N/A"),
            "답변": hit.payload.get("답변", "N/A"),
            "유사도 점수": hit.score
        }
        results.append(result)
    return results

tokenizer_config.json:   0%|          | 0.00/64.0k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/732k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/233k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.64M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/577 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/868 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [ ]:
def generate_response(query, context):
    input_text = f"질문: {query}\n컨텍스트: {context}\n답변:"
    input_ids = tokenizer.encode(input_text, return_tensors="pt")

    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_length=512,
            num_return_sequences=1,
            no_repeat_ngram_size=2,
            top_k=50,
            top_p=0.92,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(output[0], skip_special_tokens=True)
    return response.strip()

In [ ]:
def advanced_rag_search():
    # 무한 루프를 시작하여 사용자의 입력을 계속해서 받습니다.
    while True:
        # 사용자에게 질문을 입력 받습니다.
        query = input("질문을 입력하세요 (종료하려면 '끝내기' 입력): ")

        # 사용자가 '끝내기'라고 입력하면 루프를 종료합니다.
        if query.lower() == '끝내기':
            print("검색을 종료합니다.")
            break

        # 질병 정보를 검색합니다.
        results = search_disease_info(query)

        # 검색 결과가 있는지 확인합니다.
        if results:
            best_match = results[0]  # 가장 유사도가 높은 결과 선택
            context = best_match['답변']  # 관련된 답변을 가져옵니다.
            print(f"\n'{query}'에 대한 검색 결과:\n")
            print(f"관련 질병: {best_match['질병']}")  # 관련 질병 출력
            print(f"유사도 점수: {best_match['유사도 점수']:.4f}")  # 유사도 점수 출력
            print("원본 답변:")
            print(context)  # 원본 답변 출력

            # DB의 답변을 바탕으로 생성된 답변을 출력합니다.
            print("\nko-gpt-trinity-cw 생성 답변:")
            generated_response = generate_response(query, context)
            print(generated_response)
        else:
            # 검색 결과가 없는 경우 메시지를 출력합니다.
            print("검색 결과가 없습니다.")

        # 각 질문과 응답 사이에 구분선을 출력합니다.
        print("-" * 50)

# 함수를 호출하여 실행합니다.
advanced_rag_search()


질문을 입력하세요 (종료하려면 '끝내기' 입력): 운동 전후 종아리 뒤쪽 통증이 있고 발목을 움직일때 소리가 나

'운동 전후 종아리 뒤쪽 통증이 있고 발목을 움직일때 소리가 나'에 대한 검색 결과:

관련 질병: 아킬레스 건염
유사도 점수: 0.8145
원본 답변:
아킬레스 건염은 주로 발목 부근에서 발생하는 염증성 질환으로 알려져 있습니다. 아킬레스 건염은 발목 주변에서 통증과 부종을 동반하는 통증이 주로 나타납니다. 운동과 같은 활동 후 발뒤꿈치의 아킬레스 건 부분에서 염증이 발생하면 발목 주변에 강한 통증이 생길 수 있습니다. 또한, 아킬레스 건염은 종아리로 통증이 전해질 수 있어 발목 부분을 걷거나 달리는 동안 통증이 느껴질 수 있습니다. 만약 아킬레스 건염을 의심한다면, 휴식을 취해도 통증이 지속되는 경우 의사의 진료를 받는 것이 중요합니다. 아킬레스 건염은 다양한 증상을 동반하는 염증성 질환입니다. 만약 발목 주변에서 통증이 나타나면 의사의 진료를 받아야 합니다. 조기에 대처하지 않으면 심각한 합병증을 초래할 수 있으므로, 가능한 빠른 시일 내에 증상을 인지하고 의사의 도움을 받는 것이 중요합니다.

ko-gpt-trinity-cw 생성 답변:
질문: 운동 전후 종아리 뒤쪽 통증이 있고 발목을 움직일때 소리가 나
컨텍스트: 아킬레스 건염은 주로 발목 부근에서 발생하는 염증성 질환으로 알려져 있습니다. 아킬레스 건염은 발목 주변에서 통증과 부종을 동반하는 통증이 주로 나타납니다. 운동과 같은 활동 후 발뒤꿈치의 아킬레스 건 부분에서 염증이 발생하면 발목 주변에 강한 통증이 생길 수 있습니다. 또한, 아킬레스 건염은 종아리로 통증이 전해질 수 있어 발목 부분을 걷거나 달리는 동안 통증이 느껴질 수 있습니다. 만약 아킬레스 건염을 의심한다면, 휴식을 취해도 통증이 지속되는 경우 의사의 진료를 받는 것이 중요합니다. 아킬레스 건염은 다양한 증상을 동반하는 염증성 질환입니다. 만약 발목 주변에서 통증이 나타나면 의사의 진료를 받아야 합니다. 조기에 대처하지